In [52]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
from torchvision import transforms
import kagglehub

# Dataset Dataloader

In [53]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


In [54]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device :{device}")

Device :cuda


In [55]:
Train_path = os.path.join(path,"PlantVillage","train")
Test_path = os.path.join(path,"PlantVillage","val")

In [56]:
print("Train Path :",Train_path)

Train Path : /kaggle/input/plantvillage/PlantVillage/train


# Transformation

In [57]:
transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3,std=[0.5]*3)
    ]
)

# Custom Dataset

In [58]:
from genericpath import isfile

class MultiClassClassifier(Dataset):
  def __init__(self,root_dir,transform=None):
    super().__init__()
    self.samples = []
    self.transform = transform
    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

    self.class_to_idx = {class_name : idx for idx,class_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)
      for img in os.listdir(class_path):
        img_path = os.path.join(class_path,img)
        label = self.class_to_idx[class_name]

        if os.path.isfile(img_path):
          self.samples.append((img_path,label))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):

    img_path,label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image,label


In [60]:
train_dataset = MultiClassClassifier(Train_path,transform)
test_dataset = MultiClassClassifier(Test_path,transform)

num_classes = len(train_dataset.classes)
print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset))
print("Full test size:", len(test_dataset))

Number of classes: 38
Full train size: 43444
Full test size: 10861


Subset for testing on small data

In [61]:
#train_dataset = Subset(train_dataset_full,list(range(min(10000,len(train_dataset_full)))))
#test_dataset = Subset(test_dataset_full,list(range(min(2000,len(test_dataset_full)))))

#DataLoader

In [62]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle=True,pin_memory=pin)
test_loader = DataLoader(test_dataset,batch_size = 32,shuffle=False,pin_memory=pin)

#CNN

In [63]:
class MyCNN(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),
        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),
        nn.Conv2d(64,128,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*16*16,128),
        nn.ReLU(),
        nn.Dropout(0.4),#40% neuron will be off randomly to avoid overfitting
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64,num_classes)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)

    return x

In [64]:
model = MyCNN(num_classes=num_classes).to(device)

# Training Setup

In [65]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop

In [66]:
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for batch_features,batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  avg_loss = total_loss/len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 1.9233
Epoch 2/10, Loss: 1.2358
Epoch 3/10, Loss: 0.9597
Epoch 4/10, Loss: 0.7923
Epoch 5/10, Loss: 0.6579
Epoch 6/10, Loss: 0.5728
Epoch 7/10, Loss: 0.5107
Epoch 8/10, Loss: 0.4698
Epoch 9/10, Loss: 0.4219
Epoch 10/10, Loss: 0.3957


# Evaluation Loop

In [67]:
total,correct = 0,0
model.eval()
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    _,predicted = torch.max(outputs,1)
    total += len(batch_labels)
    correct += (predicted==batch_labels).sum().item()

Test_accuracy = (correct/total)*100
print(f"Test Accuracy : {Test_accuracy:.2f}%")

Test Accuracy : 94.56%
